# money

**Caution:** I have not checked this dataset for errors. Don't trust it! I'm just using it for `pandas` and `matplotlib` examples.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('bmh')

In [ ]:
# paths
DATA = Path.cwd() / "data"
MONEY = DATA / "prices/money.csv"

# plot kwargs
plotkw = {
    "figsize": (9, 3),
    "grid": True,
    "title": "exchange rate",
    "ylabel": "USD",
}


In [ ]:
def clean_cols(data):
    return [x.lower().replace(' ', '_') for x in data.columns]

In [ ]:
data = pd.read_csv(MONEY)
data.columns = clean_cols(data)
data['date'] = pd.to_datetime(data['date'])
data = (
    data
    .set_index(['date', 'country'])
    ['exchange_rate']
    .unstack('country')
    .sort_index()
    .sort_index(axis=1)
)
data.columns = clean_cols(data)
data.tail()

In [ ]:
data.columns

In [ ]:
# Many European countries switched to EUR.
# Their exchange rates will be NaN after switching.
badrows = data.isnull().sum()
badrows.nlargest(10)

In [ ]:
# Most of these rates are in units of 1 / USD.
# Some rates are in units of USD. This is confusing.
# Let's convert them all to USD.
# see https://github.com/datasets/exchange-rates/tree/main
flipped = [
    'euro',
    'ireland',
]
flipcols = sorted(set(data.columns) - set(flipped))
data[flipcols] = 1 / data[flipcols]
data[flipped].plot(**plotkw)

In [ ]:
# Hong Kong has its own currency HKD (Hong Kong Dollar).
# In 1983-10-07, it was "pegged" at about 0.1282 USD.
# Since then, the hong_kong plot should be a flat line.
data['hong_kong'].plot(**plotkw)

In [ ]:
# 人民币 was "pegged" to about 0.115 USD from 1994 until 2005.
# Since then, it has "floated" in a narrow range
# based on a "basket" of other currencies.
data['china'].plot(**plotkw)

In [ ]:
# Show the most recent not-null rates
(data
    .loc[data.index.max()]
    .dropna()
    .sort_values()
    .plot(**plotkw, kind='bar')
)